In [ ]:
import os
api_key = os.getenv("OPENAI_API_KEY")
print("Key set!")

Key set!


In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()  # reads from .env file

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Client ready!")

Client ready!


In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for information on a topic",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("Tools defined!")

Tools defined!


In [6]:
def web_search(query: str) -> str:
    # Fake results for now — just to test the agent loop works
    # We'll replace this with real search in the next phase
    return f"""
    Search results for: '{query}'
    
    Result 1: AI agents are autonomous systems that can perceive, decide, and act.
    Result 2: Modern AI agents use LLMs as their brain and tools to interact with the world.
    Result 3: Multi-agent systems have multiple AI agents working together on complex tasks.
    """

print("Search function ready!")

Search function ready!


In [10]:
def run_agent(user_question: str):
    print(f"\n🔍 Question: {user_question}")
    print("-" * 50)
    
    # conversation history
    messages = [
        {"role": "system", "content": "You are a research assistant. Use the web_search tool to find information and answer the user's question."},
        {"role": "user", "content": user_question}
    ]
    
    # the loop
    while True:
        # send to GPT
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            tools=tools,
            messages=messages
        )
        
        choice = response.choices[0]
        
        # did GPT want to use a tool?
        if choice.finish_reason == "tool_calls":
            tool_call = choice.message.tool_calls[0]
            query = json.loads(tool_call.function.arguments)["query"]
            
            print(f"🛠️  Agent is searching for: {query}")
            
            # run the tool
            result = web_search(query)
            
            print(f"📄 Got results, thinking...")
            
            # add to conversation history
            messages.append(choice.message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        
        # GPT is done, has a final answer
        else:
            final_answer = choice.message.content
            print(f"\n✅ Answer:\n{final_answer}")
            return final_answer

print("Agent loop ready!")

Agent loop ready!


In [11]:
run_agent("What are AI agents and how do they work?")


🔍 Question: What are AI agents and how do they work?
--------------------------------------------------


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-*******************************************************************************************************************************************************hdZQ. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
X